# ETL Pipeline Doctor — GRPO Training Notebook

This notebook trains `Qwen/Qwen3-0.6B` on the ETL Pipeline Doctor RL environment using GRPO.

**Requirements:** Colab Pro (H100 GPU) or Vultr H100.

**Expected reward trajectory:**
- Steps 1–20: r_mean ≈ −1.5 (agent does not know tools)
- Steps 20–80: r_mean climbs to ~0.5 as tool usage improves
- Steps 80–200: r_mean climbs to ~+1.5 as diagnosis strategy emerges

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Install Ollama and pull gemma4:26b
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
import threading

threading.Thread(
    target=lambda: subprocess.run(["ollama", "serve"], check=False), daemon=True
).start()
import time

time.sleep(3)
!ollama pull gemma4:26b

In [ ]:
# Clone the repo (replace with your fork URL)
!git clone https://github.com/YOUR_USERNAME/etl-pipeline-doctor.git
%cd etl-pipeline-doctor

In [ ]:
# Install dependencies
!uv sync --extra train

In [ ]:
# Smoke test (no GPU needed)
!uv run python train.py --dry-run

In [ ]:
# Full training run (200 steps on H100)
!uv run python train.py --num-generations 4 --max-steps 200

In [ ]:
# Plot reward curve
!uv run python plot_rewards.py training/grpo-output/metrics.jsonl

In [ ]:
# Show reward curve
from IPython.display import Image

Image("reward_curve.png")

In [ ]:
# Evaluate base vs trained
!uv run python eval.py --checkpoint training/grpo-output/checkpoint-200

In [ ]:
# Show evaluation results
with open("eval_results.md") as f:
    print(f.read())

Image("eval_plot.png")